# Gravitational Lens Classification (PyTorch)

This notebook provides a clean end-to-end binary classification pipeline for gravitational lens detection, suitable for project submission.

## 1. Imports

Import core libraries and project modules from `src/`.

In [ ]:
from __future__ import annotations

import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18

## 2. Dataset Class (from data.py)

Define the custom dataset and helper functions used to read lens and non-lens samples.

In [ ]:
VALID_EXTENSIONS = {'.npy', '.npz', '.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}

@dataclass(frozen=True)
class Sample:
    path: Path
    label: int

class LensDataset(Dataset):
    """Dataset for gravitational lens binary classification."""

    def __init__(self, samples: Sequence[Sample], transform=None) -> None:
        self.samples = list(samples)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, int]:
        sample = self.samples[index]
        image = self._load_file(sample.path)
        tensor = torch.from_numpy(image)
        if self.transform is not None:
            tensor = self.transform(tensor)
        return tensor, sample.label

    @staticmethod
    def _load_file(path: Path) -> np.ndarray:
        suffix = path.suffix.lower()
        if suffix == '.npy':
            arr = np.load(path)
        elif suffix == '.npz':
            data = np.load(path)
            arr = data[data.files[0]]
        else:
            from PIL import Image
            arr = np.asarray(Image.open(path).convert('RGB'))
        return _to_chw_float32(arr)

def _to_chw_float32(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 2:
        arr = np.expand_dims(arr, axis=0)
    elif arr.ndim == 3:
        if arr.shape[0] == 3:
            pass
        elif arr.shape[-1] == 3:
            arr = np.transpose(arr, (2, 0, 1))
        else:
            raise ValueError(f'Unsupported 3D image shape: {arr.shape}')
    else:
        raise ValueError(f'Unsupported image shape: {arr.shape}')

    arr = arr.astype(np.float32)
    if arr.max() > 1.0:
        arr = arr / 255.0
    return arr

def collect_samples(class_dir: Path, label: int) -> List[Sample]:
    if not class_dir.exists():
        raise FileNotFoundError(f'Missing directory: {class_dir}')

    samples: List[Sample] = []
    for path in sorted(class_dir.rglob('*')):
        if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS:
            samples.append(Sample(path=path, label=label))

    if not samples:
        raise ValueError(f'No supported files found in: {class_dir}')
    return samples

def build_train_test_samples(data_root: Path) -> Tuple[List[Sample], List[Sample]]:
    train_lens = collect_samples(data_root / 'train_lenses', label=1)
    train_nonlens = collect_samples(data_root / 'train_nonlenses', label=0)
    test_lens = collect_samples(data_root / 'test_lenses', label=1)
    test_nonlens = collect_samples(data_root / 'test_nonlenses', label=0)

    train_samples = train_lens + train_nonlens
    test_samples = test_lens + test_nonlens
    return train_samples, test_samples

## 3. DataLoader setup

Create train, validation, and test dataloaders with augmentation and class-weight calculation for imbalance handling.

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def create_synthetic_dataset(root: Path, n_train: int = 80, n_test: int = 20) -> None:
    # Fallback so the notebook runs end-to-end even without real dataset folders.
    folder_specs = [
        ('train_lenses', 1, n_train // 2),
        ('train_nonlenses', 0, n_train // 2),
        ('test_lenses', 1, n_test // 2),
        ('test_nonlenses', 0, n_test // 2),
    ]
    for folder_name, label, n_samples in folder_specs:
        folder = root / folder_name
        folder.mkdir(parents=True, exist_ok=True)
        if list(folder.glob('*.npy')):
            continue
        for i in range(n_samples):
            img = np.random.rand(3, 64, 64).astype(np.float32)
            if label == 1:
                img[:, 24:40, 24:40] += 0.35
                img = np.clip(img, 0.0, 1.0)
            np.save(folder / f'sample_{i:04d}.npy', img)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

project_root = Path.cwd().resolve().parent
data_root = project_root
required_dirs = ['train_lenses', 'train_nonlenses', 'test_lenses', 'test_nonlenses']
if not all((data_root / d).exists() for d in required_dirs):
    print('Dataset folders not found. Creating synthetic dataset for notebook run...')
    create_synthetic_dataset(data_root)

train_samples_all, test_samples = build_train_test_samples(data_root)
labels_all = [s.label for s in train_samples_all]
train_idx, val_idx = train_test_split(
    np.arange(len(train_samples_all)),
    test_size=0.2,
    random_state=42,
    stratify=labels_all,
 )
train_samples = [train_samples_all[i] for i in train_idx]
val_samples = [train_samples_all[i] for i in val_idx]

train_transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = LensDataset(train_samples, transform=train_transform)
val_dataset = LensDataset(val_samples, transform=eval_transform)
test_dataset = LensDataset(test_samples, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

train_labels = torch.tensor([s.label for s in train_samples], dtype=torch.long)
class_counts = torch.bincount(train_labels, minlength=2)
class_weights = (len(train_labels) / (2.0 * class_counts.float())).to(device)

print(f'Train={len(train_dataset)} | Val={len(val_dataset)} | Test={len(test_dataset)}')
print(f'Class counts: {class_counts.tolist()}')
print(f'Class weights: {class_weights.detach().cpu().tolist()}')

## 4. Model Definition (from model.py)

Define ResNet18 and replace the final layer for binary classification.

In [ ]:
def build_resnet18_binary(pretrained: bool = True) -> nn.Module:
    weights = ResNet18_Weights.DEFAULT if pretrained else None
    model = resnet18(weights=weights)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 2)
    return model

model = build_resnet18_binary(pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

print(model.__class__.__name__)
print('Model initialized successfully.')

## 5. Training Loop (from train.py)

Train the model and print epoch-wise training and validation metrics.

In [ ]:
@torch.no_grad()
def evaluate_loader(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> Dict[str, np.ndarray | float]:
    model.eval()
    all_labels, all_scores = [], []
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        probs = torch.softmax(logits, dim=1)[:, 1]

        running_loss += loss.item() * images.size(0)
        all_labels.append(labels.detach().cpu().numpy())
        all_scores.append(probs.detach().cpu().numpy())

    y_true = np.concatenate(all_labels)
    y_score = np.concatenate(all_scores)
    auc = float('nan')
    if len(np.unique(y_true)) > 1:
        auc = float(roc_auc_score(y_true, y_score))

    return {
        'loss': running_loss / len(loader.dataset),
        'auc': auc,
        'y_true': y_true,
        'y_score': y_score,
    }

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)

num_epochs = 5
history: List[Dict[str, float]] = []

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = evaluate_loader(model, val_loader, criterion, device)
    history.append({
        'epoch': float(epoch),
        'train_loss': train_loss,
        'val_loss': float(val_metrics['loss']),
        'val_auc': float(val_metrics['auc']),
    })
    print(
        f"Epoch {epoch:02d}/{num_epochs} | train_loss={train_loss:.6f} | "
        f"val_loss={float(val_metrics['loss']):.6f} | val_auc={float(val_metrics['auc']):.6f}"
    )

## 6. Evaluation (ROC-AUC)

Evaluate the trained model on the test set and report the final AUC score.

In [ ]:
test_metrics = evaluate_loader(model, test_loader, criterion, device)
y_true_test = test_metrics['y_true']
y_score_test = test_metrics['y_score']
final_auc = float(test_metrics['auc'])

print(f"Final Test Loss: {float(test_metrics['loss']):.6f}")
print(f"Final Test ROC-AUC: {final_auc:.6f}")

fpr, tpr, _ = roc_curve(y_true_test, y_score_test)

## 7. ROC Curve Plot

Plot the ROC curve and save the figure for reporting.

In [ ]:
def plot_roc(fpr: np.ndarray, tpr: np.ndarray, auc: float, output_path: Path) -> None:
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, linewidth=2, label=f"ResNet18 (AUC = {auc:.4f})")
    plt.plot([0, 1], [0, 1], linestyle='--', linewidth=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve - Gravitational Lens Classification')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(output_path, dpi=200)
    plt.show()
    plt.close()

outputs_dir = project_root / 'outputs'
outputs_dir.mkdir(parents=True, exist_ok=True)
roc_path = outputs_dir / 'roc_curve_notebook_submission.png'
plot_roc(fpr, tpr, final_auc, roc_path)
print(f"Saved ROC plot to: {roc_path}")